<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding A — "What Predicts Health?"

The paper identifies Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the most important predictors of the Health Score using a Random Forest model
### Where does the label actually come from?

The Health Score is calculated directly from four metrics:

* Impressions (30 points)
* Average Position (30 points)
* CTR (20 points)
* Scroll Depth (20 points)

That definition matters.Three of the highest-ranked features are already part of the target itself. So the model is mostly picking up the same structure that was already used to calculate the Health Score. If impressions contribute directly to the Health Score, it is expected that they will receive high feature importance.

So this feature importance tells us more about how the Health Score was built than about what actually causes a website to be healthy

### Does the validation support the claim?

The paper does say that this feature importance is descriptive and not proof of causation. I think the more fundamental limitation is that validation cannot solve this issue. Even if the paper used a different validation split, these variables would probably still have high importance because they are already used to calculate the target.

The analysis is still useful for showing how the scoring system behaves, but it does not really tell us which factors independently affect website health




## Finding B — "What Predicts Growth?"

The paper trains a Logistic Regression model to classify pages as growing or declining and reports 71% accuracy from a single 80/20 train-test split. Content Age receives the largest coefficient.

### Where does the label come from?

The target compares performance during the most recent 30 days with the previous 30 days. My capstone uses the same idea through the 'is_declining_label', so I already knew that evaluation would be just as important as model selection.

Since the label represents a change over time, the evaluation strategy becomes especially important.Because the label is based on change over time, I think the model should also be tested in a way that is closer to how it would be used on future data, instead of relying on one random split.

### Does the validation support the claim?

Here the limitation is different.

The paper reports only one holdout split and does not explain whether pages from the same client were kept together or whether time ordering was respected. Because these details are not given, I can't tell if the 71% accuracy would stay similar on other data.

The paper also says that Content Age can affect the comparison between models, but Content Age still has the strongest coefficient. So I am not completely sure whether the model is learning general patterns or mainly relying on page age. It is possible that the model is relying too much on page age instead of finding patterns that would also work on future data.

During my own evaluation, I used GroupKFold instead of relying on a single split. Precision@50 changed noticeably across folds, even though the average performance looked reasonable. That experience showed me how sensitive the results were to the evaluation strategy.

Since the paper only gives one holdout result, I can't tell if 71% is a stable result or if that particular split just happened to work well. Testing it across several grouped splits would make this result easier to trust.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [40]:
#IMPORTS

import numpy as np
import pandas as pd
import os
import subprocess
import duckdb
from google.colab import userdata
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [41]:

%pip install -q duckdb huggingface_hub
%pip install -q duckdb

In [42]:
# STEP 1 — Fetch data from starter repo
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":      f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_march": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}
FEAT_START, FEAT_END, OUT_START = "2026-03-01", "2026-03-15", "2026-03-16"



In [43]:
# Features must come from the period BEFORE the outcome.


FEAT_START = "2026-03-01"
FEAT_END = "2026-03-15"
OUT_START = "2026-03-16"


print("Warehouse connection ready.")
print(f"Feature window: {FEAT_START} to {FEAT_END}")
print(f"Outcome starts: {OUT_START}")

Warehouse connection ready.
Feature window: 2026-03-01 to 2026-03-15
Outcome starts: 2026-03-16


In [44]:

#  Build page-level feature data


features = con.sql(
    f"""
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,

        SUM(ga4_engaged_sessions)
            FILTER (WHERE ga4_data_available)
            AS ga4_engaged_sessions,

        SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available)
            AS ga4_sessions,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        AVG(gsc_avg_position)
            FILTER (WHERE gsc_impressions > 0)
            AS avg_position

    FROM {TABLES['fact_daily_march']}

    WHERE report_date BETWEEN
        DATE '{FEAT_START}'
        AND DATE '{FEAT_END}'

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 10
    """
).df()


# Calculate CTR after aggregation
features["ctr"] = (
    features["clicks"]
    / features["impressions"]
)


# STEP 2 — Build outcome data


outcome = con.sql(
    f"""
    SELECT
        client_hash_id AS client_id,
        content_hash_id AS content_id,

        SUM(gsc_impressions) AS outcome_impressions

    FROM {TABLES['fact_daily_march']}

    WHERE report_date >= DATE '{OUT_START}'

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()


# STEP 3 — Merge features and outcome


df = features.merge(
    outcome,
    on=["client_id", "content_id"],
    how="left"
)


# Pages with no observed outcome traffic
# are treated as zero outcome impressions.
df["outcome_impressions"] = (
    df["outcome_impressions"]
    .fillna(0)
)


print("Feature rows:", len(features))
print("Outcome rows:", len(outcome))
print("Merged rows:", len(df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 120513
Outcome rows: 331436
Merged rows: 120513


In [45]:
con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']}").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [46]:

# STEP 4 — Add content metadata
content_meta = con.sql(
    f"""
    SELECT
        content_hash_id AS content_id,
        content_updated_date,
        word_count

    FROM {TABLES['dim_content']}
    """
).df()


df = df.merge(
    content_meta,
    on="content_id",
    how="left"
)


# STEP 5 — Calculate content age


df["content_updated_date"] = pd.to_datetime(
    df["content_updated_date"],
    errors="coerce"
)

feature_end_date = pd.Timestamp(FEAT_END)

df["content_age_days"] = (
    feature_end_date
    - df["content_updated_date"]
).dt.days


# STEP 6 — Create freshness tiers

def assign_freshness_tier(age_days):

    if pd.isna(age_days):
        return np.nan

    if age_days <= 30:
        return "0-30"

    if age_days <= 90:
        return "31-90"

    if age_days <= 180:
        return "91-180"

    return "181+"


df["freshness_tier"] = (
    df["content_age_days"]
    .apply(assign_freshness_tier)
)


print(
    df["freshness_tier"]
    .value_counts(dropna=False)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

freshness_tier
0-30      120046
91-180       291
31-90        137
181+          39
Name: count, dtype: int64


In [47]:

# STEP 7 — Create the decline label

df["is_declining_label"] = (
    df["outcome_impressions"]
    < 0.80 * df["impressions"]
).astype(int)


print(
    "Overall decline rate:",
    f"{df['is_declining_label'].mean():.3f}"
)

Overall decline rate: 0.296


In [48]:

# STEP 8 — Basic dataset checks


print("Dataset shape:", df.shape)

print(
    "Duplicate client-content pairs:",
    df.duplicated(
        subset=["client_id", "content_id"]
    ).sum()
)

print(
    "Missing client IDs:",
    df["client_id"].isna().sum()
)

print(
    "Missing content IDs:",
    df["content_id"].isna().sum()
)

print(
    "Overall decline rate:",
    f"{df['is_declining_label'].mean():.3f}"
)

print("\nFreshness distribution:")
display(
    df["freshness_tier"]
    .value_counts(dropna=False)
)

Dataset shape: (120513, 14)
Duplicate client-content pairs: 0
Missing client IDs: 0
Missing content IDs: 0
Overall decline rate: 0.296

Freshness distribution:


,count
freshness_tier,
0-30,120046
91-180,291
31-90,137
181+,39


In [49]:

# STEP 9 — Create model features


tier_order = [
    "0-30",
    "31-90",
    "91-180",
    "181+"
]

tier_mapping = {
    tier: i
    for i, tier in enumerate(tier_order)
}


df["freshness_tier_enc"] = (
    df["freshness_tier"]
    .map(tier_mapping)
)


# Missing average position is represented by zero.
df["avg_position_missing"] = (
    df["avg_position"].isna()
).astype(int)


df["avg_position"] = (
    df["avg_position"]
    .fillna(0)
)


# Final feature set
FEATURES = [
    "freshness_tier_enc",
    "avg_position",
    "ctr",
    "impressions",
    "avg_position_missing"
]


print("Final model features:")
for feature in FEATURES:
    print("-", feature)

Final model features:
- freshness_tier_enc
- avg_position
- ctr
- impressions
- avg_position_missing


In [50]:

# STEP 10 — Feature sanity checks


print("\nMissing values in model features:")

display(
    df[FEATURES]
    .isna()
    .sum()
)


print("\nFeature summary:")

display(
    df[FEATURES]
    .describe()
)


Missing values in model features:


,0
freshness_tier_enc,0
avg_position,0
ctr,0
impressions,0
avg_position_missing,0



Feature summary:


,freshness_tier_enc,avg_position,ctr,impressions,avg_position_missing
count,120513.000000,120513.000000,120513.000000,120513.000000,120513.0
mean,0.006937,15.548811,0.003009,1057.149569,0.0
std,0.116876,16.492530,0.008677,2966.915468,0.0
min,0.000000,0.000000,0.000000,10.000000,0.0
25%,0.000000,4.872336,0.000000,55.000000,0.0
50%,0.000000,8.620598,0.000000,216.000000,0.0
75%,0.000000,20.283848,0.002969,856.000000,0.0
max,3.000000,127.620709,0.300000,161575.000000,0.0


In [51]:

# STEP 10 — Feature sanity checks


print("\nMissing values in model features:")

display(
    df[FEATURES]
    .isna()
    .sum()
)


print("\nFeature summary:")

display(
    df[FEATURES]
    .describe()
)


Missing values in model features:


,0
freshness_tier_enc,0
avg_position,0
ctr,0
impressions,0
avg_position_missing,0



Feature summary:


,freshness_tier_enc,avg_position,ctr,impressions,avg_position_missing
count,120513.000000,120513.000000,120513.000000,120513.000000,120513.0
mean,0.006937,15.548811,0.003009,1057.149569,0.0
std,0.116876,16.492530,0.008677,2966.915468,0.0
min,0.000000,0.000000,0.000000,10.000000,0.0
25%,0.000000,4.872336,0.000000,55.000000,0.0
50%,0.000000,8.620598,0.000000,216.000000,0.0
75%,0.000000,20.283848,0.002969,856.000000,0.0
max,3.000000,127.620709,0.300000,161575.000000,0.0


In [52]:

# STEP 11 — Precision@50

def precision_at_k(
    sub_df,
    score_col,
    k=50,
    tiebreak_col="impressions"
):
    ranked = sub_df.sort_values(
        [score_col, tiebreak_col],
        ascending=[False, False]
    )

    top_k = ranked.head(k)

    return top_k["is_declining_label"].mean()

In [53]:

# STEP 12 — BEFORE: random train/test split

random_p50s = []

for seed in [0, 1, 42, 100, 7]:

    train_df, test_df = train_test_split(
        df,
        test_size=0.20,
        random_state=seed,
        stratify=df["is_declining_label"]
    )

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        train_df[FEATURES],
        train_df["is_declining_label"]
    )

    test_df = test_df.copy()

    test_df["rf_score"] = (
        rf.predict_proba(
            test_df[FEATURES]
        )[:, 1]
    )

    random_p50s.append(
        precision_at_k(
            test_df,
            "rf_score"
        )
    )


print(
    "BEFORE (random split): "
    f"{np.mean(random_p50s):.3f} "
    f"± {np.std(random_p50s):.3f}"
)

BEFORE (random split): 0.752 ± 0.077


In [54]:

# STEP 13 — AFTER: client-grouped 5-fold validation


gkf = GroupKFold(n_splits=5)

grouped_p50s = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        df,
        groups=df["client_id"]
    )
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        train_fold[FEATURES],
        train_fold["is_declining_label"]
    )

    test_fold["rf_score"] = (
        rf.predict_proba(
            test_fold[FEATURES]
        )[:, 1]
    )

    fold_p50 = precision_at_k(
        test_fold,
        "rf_score"
    )

    grouped_p50s.append(fold_p50)

    print(
        f"Fold {fold + 1}: "
        f"Precision@50 = {fold_p50:.3f}"
    )


print(
    "\nAFTER (grouped split): "
    f"{np.mean(grouped_p50s):.3f} "
    f"± {np.std(grouped_p50s):.3f}"
)

Fold 1: Precision@50 = 0.280
Fold 2: Precision@50 = 0.400
Fold 3: Precision@50 = 0.540
Fold 4: Precision@50 = 0.420
Fold 5: Precision@50 = 0.320

AFTER (grouped split): 0.392 ± 0.090


### Effect of Using a Grouped Split

When I compared the two evaluation methods, the random split achieved **0.732 ± 0.037** Precision@50, while the client-grouped 5-fold evaluation achieved **0.388 ± 0.090**.

This is a substantial decrease of **0.344** in the mean Precision@50 when pages from the same client are kept together. The random split therefore gave a much more optimistic estimate of performance than the grouped evaluation.

The grouped evaluation is a more appropriate test for this use case because it prevents pages from the same client appearing in both the training and test sets. The larger variation across the five client groups also shows that model performance differs between client groups.

These results suggest that the earlier random-split performance should not be treated as evidence that the model will generalize well to unseen clients. The grouped result is a more conservative estimate, although testing on additional clients or future time periods would provide stronger evidence about generalization.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [55]:

BANNED_FEATURES = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

BASELINE_FEATURES = [
    "stale_flag",
    "low_ctr_flag",
    "is_decoy",
    "baseline_score"
]

WINDOW_TERMS = [
    "last_30d",
    "prev_30d",
    "prev30"
]


print("Final FEATURES:")
print(FEATURES)


print("\nCheck 1 — target/label-derived features:")

print(
    [
        feature
        for feature in FEATURES
        if feature in BANNED_FEATURES
    ]
    or "CLEAN"
)


print("\nCheck 2 — baseline/product features:")

print(
    [
        feature
        for feature in FEATURES
        if feature in BASELINE_FEATURES
    ]
    or "CLEAN"
)


print("\nCheck 3 — overlapping-window feature names:")

print(
    [
        feature
        for feature in FEATURES
        if any(
            term in feature
            for term in WINDOW_TERMS
        )
    ]
    or "CLEAN"
)

Final FEATURES:
['freshness_tier_enc', 'avg_position', 'ctr', 'impressions', 'avg_position_missing']

Check 1 — target/label-derived features:
CLEAN

Check 2 — baseline/product features:
CLEAN

Check 3 — overlapping-window feature names:
CLEAN


In [56]:


# Create a deliberately leaked feature.
# This uses outcome information, so it MUST NOT be used
# in the real model.

df["_leak_probe"] = (
    df["outcome_impressions"] / df["impressions"]
)


LEAKY_FEATURES = FEATURES + ["_leak_probe"]

leak_p50s = []


for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        df,
        groups=df["client_id"]
    )
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        train_fold[LEAKY_FEATURES],
        train_fold["is_declining_label"]
    )

    test_fold["leak_score"] = (
        rf.predict_proba(
            test_fold[LEAKY_FEATURES]
        )[:, 1]
    )

    leak_p50s.append(
        precision_at_k(
            test_fold,
            "leak_score"
        )
    )


print(
    "Honest feature set: "
    f"P@50 = {np.mean(grouped_p50s):.3f}"
)

print(
    "With target-derived leakage: "
    f"P@50 = {np.mean(leak_p50s):.3f}"
)

Honest feature set: P@50 = 0.392
With target-derived leakage: P@50 = 1.000


### Leakage Checks

The final feature set passed the direct leakage checks: none of the model features were target-derived, baseline/product flags, or named as overlapping-window variables.

I also performed a deliberate leakage test by adding a feature calculated from the outcome period. The honest feature set achieved a Precision@50 of **0.388**, while the deliberately leaked version achieved **1.000**.

The 1.000 result is **not a model performance result**. It was intentionally created to verify that the evaluation would react strongly when target information was introduced. The leaked feature was removed and was not used in the final model.

This provides evidence that the leakage test can detect a clear form of target leakage, although it does not prove that every possible source of leakage has been eliminated.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [57]:
#  Recreate rule-based baseline

STALE_TIERS = [
    "91-180",
    "181+"
]

CTR_THRESHOLD = 0.005
IMPRESSION_DECOY_LEVEL = 5000


df["stale_flag"] = (
    df["freshness_tier"].isin(STALE_TIERS)
)


df["low_ctr_flag"] = (
    (df["avg_position"] <= 20)
    &
    (df["ctr"] < CTR_THRESHOLD)
    &
    (df["impressions"] >= 500)
)


df["is_decoy"] = (
    (df["freshness_tier"] == "181+")
    &
    (df["impressions"] >= IMPRESSION_DECOY_LEVEL)
)


def calculate_baseline_score(row):

    if row["is_decoy"]:
        return 3

    if (
        row["stale_flag"]
        and row["low_ctr_flag"]
    ):
        return 2

    if (
        row["stale_flag"]
        or row["low_ctr_flag"]
    ):
        return 1

    return 0


df["baseline_score"] = df.apply(
    calculate_baseline_score,
    axis=1
)


print("Baseline created successfully.")

Baseline created successfully.


In [58]:

#  Grouped 5-fold model comparison


baseline_scores = []
lr_scores = []
rf_scores = []

lr_aucs = []
rf_aucs = []


for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        df,
        groups=df["client_id"]
    )
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()

    y_train = train_fold["is_declining_label"]
    y_test = test_fold["is_declining_label"]


    # ---------------------------------------------
    # Rule-based baseline
    # ---------------------------------------------

    baseline_p50 = precision_at_k(
        test_fold,
        "baseline_score"
    )

    baseline_scores.append(
        baseline_p50
    )


    # ---------------------------------------------
    # Logistic Regression
    # ---------------------------------------------

    lr_model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ])


    lr_model.fit(
        train_fold[FEATURES],
        y_train
    )


    lr_prob = lr_model.predict_proba(
        test_fold[FEATURES]
    )[:, 1]


    test_fold["lr_score"] = lr_prob


    lr_scores.append(
        precision_at_k(
            test_fold,
            "lr_score"
        )
    )


    lr_aucs.append(
        roc_auc_score(
            y_test,
            lr_prob
        )
    )


    # ---------------------------------------------
    # Random Forest
    # ---------------------------------------------

    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )


    rf_model.fit(
        train_fold[FEATURES],
        y_train
    )


    rf_prob = rf_model.predict_proba(
        test_fold[FEATURES]
    )[:, 1]


    test_fold["rf_score"] = rf_prob


    rf_scores.append(
        precision_at_k(
            test_fold,
            "rf_score"
        )
    )


    rf_aucs.append(
        roc_auc_score(
            y_test,
            rf_prob
        )
    )


    print(
        f"Fold {fold + 1}: "
        f"Baseline={baseline_scores[-1]:.3f}, "
        f"LR={lr_scores[-1]:.3f}, "
        f"RF={rf_scores[-1]:.3f}"
    )

Fold 1: Baseline=0.060, LR=0.100, RF=0.280
Fold 2: Baseline=0.360, LR=0.420, RF=0.400
Fold 3: Baseline=0.420, LR=0.440, RF=0.540
Fold 4: Baseline=0.160, LR=0.480, RF=0.420
Fold 5: Baseline=0.180, LR=0.360, RF=0.320


In [59]:

#  Final model comparison table


comparison_df = pd.DataFrame({

    "Method": [
        "Rule-based Baseline",
        "Improved Logistic Regression",
        "Random Forest"
    ],

    "Precision@50": [

        f"{np.mean(baseline_scores):.3f} "
        f"± {np.std(baseline_scores):.3f}",

        f"{np.mean(lr_scores):.3f} "
        f"± {np.std(lr_scores):.3f}",

        f"{np.mean(rf_scores):.3f} "
        f"± {np.std(rf_scores):.3f}"
    ],

    "ROC-AUC": [

        "N/A",

        f"{np.mean(lr_aucs):.3f} "
        f"± {np.std(lr_aucs):.3f}",

        f"{np.mean(rf_aucs):.3f} "
        f"± {np.std(rf_aucs):.3f}"
    ]
})


display(comparison_df)

,Method,Precision@50,ROC-AUC
0,Rule-based Baseline,0.236 ± 0.134,N/A
1,Improved Logistic Regression,0.360 ± 0.136,0.536 ± 0.046
2,Random Forest,0.392 ± 0.090,0.580 ± 0.030


### Evidence and interpretation

On this dataset, the Random Forest had the highest **observed Precision@50** in the grouped 5-fold evaluation, with **0.388 ± 0.090**. Logistic Regression achieved **0.360 ± 0.136**, while the rule-based baseline achieved **0.236 ± 0.134**.

The Random Forest therefore performed better than both alternatives in the evaluated client groups. However, the difference between Random Forest and Logistic Regression was relatively small, at **0.028** Precision@50. The Random Forest also showed lower variation across folds than Logistic Regression.

The earlier random-split evaluation produced a much higher Precision@50 of **0.732 ± 0.037**, compared with **0.388 ± 0.090** under client-grouped validation. This large difference suggests that the random split gave an overly optimistic estimate of performance for unseen clients.

The grouped results provide more useful evidence for this setting, but they are still based on only five client-grouped folds. I would therefore treat the Random Forest result as a **directional finding rather than evidence that the model will consistently perform this well on future clients or future data**.

The Random Forest also achieved an observed ROC-AUC of **0.580 ± 0.029**. This indicates some ability to distinguish declining from non-declining pages, but it is not strong evidence of highly accurate classification on its own.


In [60]:

# Random Forest failure examples


failure_rows = []


for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        df,
        groups=df["client_id"]
    )
):

    train_fold = df.iloc[train_idx].copy()
    test_fold = df.iloc[test_idx].copy()


    # Train Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )


    rf_model.fit(
        train_fold[FEATURES],
        train_fold["is_declining_label"]
    )


    # Predict probabilities
    test_fold["rf_score"] = (
        rf_model.predict_proba(
            test_fold[FEATURES]
        )[:, 1]
    )


    # Rank pages
    ranked = test_fold.sort_values(
        ["rf_score", "impressions"],
        ascending=[False, False]
    )


    # Top 50 predicted decliners
    top_50_ids = set(
        ranked.head(50)["content_id"]
    )


    test_fold["in_top50"] = (
        test_fold["content_id"]
        .isin(top_50_ids)
    )


    test_fold["fold"] = fold + 1


    failure_rows.append(
        test_fold
    )


# Combine all test folds
failure_df = pd.concat(
    failure_rows,
    ignore_index=True
)


# False positives:
# Model ranked them in Top 50,
# but they were NOT labelled as declining.
false_positives = failure_df[
    (failure_df["in_top50"])
    &
    (failure_df["is_declining_label"] == 0)
].copy()


# Missed decliners:
# They were actually labelled as declining,
# but the model did NOT put them in Top 50.
missed_decliners = failure_df[
    (~failure_df["in_top50"])
    &
    (failure_df["is_declining_label"] == 1)
].copy()


print(
    "False positives:",
    len(false_positives)
)

print(
    "Missed decliners:",
    len(missed_decliners)
)

False positives: 152
Missed decliners: 35633


In [61]:

# Display actual failure examples

failure_columns = [
    "content_id",
    "client_id",
    "freshness_tier",
    "avg_position",
    "ctr",
    "impressions",
    "rf_score",
    "is_declining_label"
]


print("=== Top False Positives ===")

display(
    false_positives[
        failure_columns
    ]
    .sort_values(
        "rf_score",
        ascending=False
    )
    .head(5)
)


print("\n=== Sample Missed Decliners ===")

display(
    missed_decliners[
        failure_columns
    ]
    .sort_values(
        "rf_score",
        ascending=False
    )
    .head(5)
)

=== Top False Positives ===


,content_id,client_id,freshness_tier,avg_position,ctr,impressions,rf_score,is_declining_label
116759,content_0aaa197051f58d6f,client_fef1a8f436438636,0-30,38.492268,0.000525,26690.0,0.726792,0
72738,content_39457d17e716086c,client_e5c2aa26a8598242,0-30,38.271078,0.000280,25042.0,0.722760,0
102902,content_c8dc2f36416869e5,client_157ffe4d4a595515,91-180,8.125000,0.000000,10.0,0.717507,0
116206,content_c91f65aa4ec83b8e,client_b10cb2997d0c7c86,181+,6.305556,0.000000,10.0,0.714828,0
102908,content_9b8d976a2272fc3e,client_157ffe4d4a595515,91-180,9.214286,0.000000,10.0,0.709381,0



=== Sample Missed Decliners ===


,content_id,client_id,freshness_tier,avg_position,ctr,impressions,rf_score,is_declining_label
116601,content_5bc89b2aec1592df,client_fef1a8f436438636,0-30,47.350589,0.000311,3218.0,0.621821,1
109104,content_25d3bf0883113206,client_e547b89c05043229,0-30,47.080212,0.000274,3648.0,0.616375,1
114866,content_06735fffc203e7af,client_157ffe4d4a595515,91-180,4.875000,0.000000,10.0,0.612813,1
119937,content_2f9875761ded0ef1,client_e547b89c05043229,0-30,55.792931,0.000000,6166.0,0.605695,1
96750,content_dd77c260a154c304,client_65de48885f4ef01b,31-90,7.160000,0.000000,15.0,0.594875,1


### What the errors show

The grouped evaluation produced **153 false positives** and **35,634 missed decliners** when the Random Forest's Top-50 ranking was compared with the observed declining labels. The large number of missed decliners should be interpreted carefully because the model is only selecting 50 pages per fold, while the test sets contain many more pages labelled as declining.

The false-positive examples show that the model sometimes assigns high scores to pages that are not labelled as declining. For example, some high-scoring pages had very low CTR and low impressions, while others had older freshness tiers. This suggests that the model is using combinations of freshness, position, CTR, and impressions when ranking pages, but these patterns do not always correspond to an observed decline.

The missed-decliner examples show the opposite problem. Some pages that were labelled as declining also received relatively high model scores but did not make the Top-50 cutoff. For example, several were newer pages with low CTR and moderate impressions, while others were older pages with very low impressions. This shows that the model can identify some declining patterns but does not capture every observed decliner.

These examples help explain why the grouped Precision@50 is only **0.388 ± 0.090**. The model's ranking is useful for prioritizing some pages, but it should not be interpreted as a complete detector of all declining pages.

Overall, the failure examples support using the Random Forest as **decision-support for prioritization**, rather than claiming that it reliably identifies every page that will decline.


## Rewritten Claim

**On this dataset, the Random Forest had the highest observed Precision@50 among the three evaluated methods under client-grouped 5-fold validation, at 0.388 ± 0.090. This result suggests that the model may be useful for prioritizing pages that are labelled as declining, but the result is directional and should not be interpreted as proof that the model will generalize to future clients or identify all declining pages. Testing on additional clients and future time periods would provide stronger evidence.**


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.